# ConvexPi — R starter

Write a quant strategy in **R** and submit it. The grader runs your R `on_day` over a hidden
out-of-sample period and scores it with the *same* engine as Python/Julia — so the same idea earns
the same OOS Sharpe in any language.


In [ ]:
# 1. Get the exact market the grader uses (deterministic from the seed — exported via Python).
system("pip install -q convexpi-lab", intern = TRUE)
system(paste("python -c",
  shQuote("from convexpi.lab import SyntheticMarket; from convexpi.lab.multilang import export_market; export_market(SyntheticMarket(seed=42), 'market')")))


In [ ]:
# 2. Load the in-sample data to fit on.
read_mat <- function(p) as.matrix(read.csv(p, header = FALSE))
prices <- read_mat("market/train/prices.csv")
feat_names <- readLines("market/train/feature_names.txt")
features <- setNames(lapply(feat_names, function(n) read_mat(file.path("market/train/features", paste0(n, ".csv")))), feat_names)
cat("prices:", dim(prices), "| features:", paste(feat_names, collapse = ", "), "\n")


## 3. Write your strategy

Define `on_day(day, features, prices, portfolio)` returning target weights (one per stock). This is
exactly what the grader runs. Keep it as a string so you can both test it and submit it.


In [ ]:
strategy_code <- '
on_day <- function(day, features, prices, portfolio) {
  sig <- features[["mom_1m"]]          # 1-month momentum, z-scored across stocks
  sig[!is.finite(sig)] <- 0
  s <- sum(abs(sig))
  if (s > 0) sig / s else sig          # long winners / short losers, gross leverage 1
}'
eval(parse(text = strategy_code))      # define on_day locally so you can poke at it


In [ ]:
# 4. (optional) sanity-check on one in-sample day
d <- 300
w <- on_day(d, lapply(features, function(m) m[d, ]), prices[d, ], rep(0, ncol(prices)))
cat("weights sum to", sum(w), "over", length(w), "stocks\n")


## 5. Submit

Create an API key at **convexpi.ai/settings/api-keys** and set it below. Your OOS Sharpe appears on
the leaderboard within a few minutes.


In [ ]:
install.packages(c("httr", "jsonlite"), quiet = TRUE)
submit_strategy <- function(name, code, slug = "demo-fall-2026") {
  key <- Sys.getenv("CONVEXPI_API_KEY")
  body <- jsonlite::toJSON(list(slug = slug, strategyName = name, code = code, language = "r"), auto_unbox = TRUE)
  r <- httr::POST("https://www.convexpi.ai/api/submissions",
                  httr::add_headers(Authorization = paste("Bearer", key), `Content-Type` = "application/json"),
                  body = body)
  cat(httr::status_code(r), httr::content(r, "text"), "\n")
}
Sys.setenv(CONVEXPI_API_KEY = "cpk_...")   # <- your key
submit_strategy("my-r-momentum", strategy_code)
